# Week 03 Coding Practice: From Text to Feature Vectors

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/obscrivn/mynewbook/blob/master/module3/week3_coding_practice.ipynb)

In this formative activity, you will turn a small text corpus into numerical representations and inspect what each representation preserves or loses. Run the cells in order. Pause for each **Predict** or **Interpret** prompt before continuing.

## Learning objectives

By the end, you should be able to:

- connect vocabulary terms to columns in a document-term matrix;
- interpret matrix shape and individual feature values;
- compare count, binary, n-gram, and TF-IDF representations;
- validate a vectorizer workflow and identify a feature-space mismatch;
- explain why sparse lexical vectors motivate a transition to embeddings.

## 1. Setup

The activity uses only packages already available in Google Colab. The corpus is defined in the notebook, so no files or network access are required.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

pd.set_option("display.max_columns", 30)
np.set_printoptions(precision=3, suppress=True)

In [ ]:
documents = [
    "Students study data science in the library.",
    "Data science students analyze text data.",
    "The library offers quiet study space.",
    "Students analyze language data with code.",
]
document_labels = ["D1", "D2", "D3", "D4"]

for label, document in zip(document_labels, documents):
    print(f"{label}: {document}")

**Predict before running:**

1. Which terms do you expect to become features?
2. What value should the feature `data` have for D2?
3. Which information about the original sentences will a unigram count matrix lose?

## 2. Worked example: a count-based document-term matrix

`CountVectorizer` lowercases and tokenizes the documents, builds a vocabulary, and counts each vocabulary term in each document. Rows represent documents; columns represent features.

In [ ]:
count_vectorizer = CountVectorizer()
count_matrix = count_vectorizer.fit_transform(documents)
feature_names = count_vectorizer.get_feature_names_out()

count_df = pd.DataFrame(
    count_matrix.toarray(),
    index=document_labels,
    columns=feature_names,
)
print(f"Matrix shape: {count_df.shape[0]} documents x {count_df.shape[1]} features")
count_df

In [ ]:
# Transparent checks: these should pass if we interpreted the matrix correctly.
assert count_df.shape == (4, 15)
assert count_df.loc["D2", "data"] == 2
assert count_df.loc["D3", "science"] == 0
print("Count-matrix checks passed.")

**Interpret:**

- Explain the values at D2–`data` and D3–`science` in words.
- Why does the matrix contain many zeros?
- Could you reconstruct the exact word order of D1 from its row? Why or why not?

## 3. Guided comparison: counts versus binary occurrence

A count representation preserves term frequency. With `binary=True`, every nonzero count becomes 1, so the representation preserves presence or absence but not repetition.

In [ ]:
binary_vectorizer = CountVectorizer(binary=True)
binary_matrix = binary_vectorizer.fit_transform(documents)
binary_df = pd.DataFrame(
    binary_matrix.toarray(),
    index=document_labels,
    columns=binary_vectorizer.get_feature_names_out(),
)

count_binary_comparison = pd.DataFrame({
    "count": count_df.loc["D2", ["data", "science", "text"]],
    "binary": binary_df.loc["D2", ["data", "science", "text"]],
})
assert count_binary_comparison.loc["data", "count"] == 2
assert count_binary_comparison.loc["data", "binary"] == 1
count_binary_comparison

**Interpret:** For what kind of analysis might repeated mentions matter? When might simple presence or absence be preferable? Give one reason for each choice.

## 4. N-grams preserve limited local order

A unigram is one token; a bigram is a sequence of two adjacent tokens. Predict which bigrams occur in more than one document before running the next cell.

In [ ]:
bigram_vectorizer = CountVectorizer(ngram_range=(2, 2))
bigram_matrix = bigram_vectorizer.fit_transform(documents)
bigram_df = pd.DataFrame(
    bigram_matrix.toarray(),
    index=document_labels,
    columns=bigram_vectorizer.get_feature_names_out(),
)

document_frequency = (bigram_df > 0).sum(axis=0)
repeated_bigrams = document_frequency[document_frequency > 1].index.tolist()
print(f"Bigram matrix shape: {bigram_df.shape}")
print("Bigrams found in more than one document:", repeated_bigrams)
assert "data science" in bigram_df.columns
assert "students analyze" in repeated_bigrams
bigram_df.loc[:, repeated_bigrams]

In [ ]:
# Guided modification: include both unigrams and bigrams.
combined_vectorizer = CountVectorizer(ngram_range=(1, 2))
combined_matrix = combined_vectorizer.fit_transform(documents)
print("Unigram-only shape:     ", count_matrix.shape)
print("Unigram + bigram shape:", combined_matrix.shape)
assert combined_matrix.shape[1] > count_matrix.shape[1]

**Interpret:** What local information did bigrams add? What sentence-level information is still missing? Explain the cost of the larger feature space.

## 5. TF-IDF: weight terms by corpus-level rarity

TF-IDF starts from term frequency and reduces the influence of terms that appear in many documents. scikit-learn also applies L2 normalization by default, so values reflect both rarity and the document vector's length.

In [ ]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    index=document_labels,
    columns=tfidf_vectorizer.get_feature_names_out(),
).round(3)

idf_values = pd.Series(
    tfidf_vectorizer.idf_,
    index=tfidf_vectorizer.get_feature_names_out(),
    name="idf",
).sort_values()

assert tfidf_df.shape == count_df.shape
assert idf_values["data"] < idf_values["quiet"]
print("IDF values, from common to rare:")
display(idf_values.to_frame().round(3))
print("TF-IDF document-term matrix:")
tfidf_df

In [ ]:
selected_terms = ["data", "science", "students", "text"]
d2_comparison = pd.DataFrame({
    "D2 count": count_df.loc["D2", selected_terms],
    "D2 TF-IDF": tfidf_df.loc["D2", selected_terms],
})
d2_comparison

**Interpret:**

- Why is `quiet` assigned a larger IDF than `data`?
- In D2, `data` has the largest raw count. Does it automatically have the largest TF-IDF weight? Explain using both corpus rarity and normalization.

### Inspecting `smooth_idf`

Smoothing behaves as if one extra document contained every term once. It avoids zero divisions and makes extreme IDF values less extreme. We set `norm=None` below to isolate IDF from vector-length normalization.

In [ ]:
smooth_vectorizer = TfidfVectorizer(norm=None, smooth_idf=True)
unsmoothed_vectorizer = TfidfVectorizer(norm=None, smooth_idf=False)
smooth_vectorizer.fit(documents)
unsmoothed_vectorizer.fit(documents)

smoothing_comparison = pd.DataFrame({
    "document frequency": (count_df > 0).sum(axis=0),
    "smoothed IDF": pd.Series(smooth_vectorizer.idf_, index=smooth_vectorizer.get_feature_names_out()),
    "unsmoothed IDF": pd.Series(unsmoothed_vectorizer.idf_, index=unsmoothed_vectorizer.get_feature_names_out()),
}).sort_values(["document frequency", "smoothed IDF"], ascending=[False, True])
assert (smoothing_comparison["smoothed IDF"] <= smoothing_comparison["unsmoothed IDF"]).all()
smoothing_comparison.round(3)

**Interpret:** Compare one common term and one rare term. Which changes more when smoothing is disabled? Why?

## 6. Critique and validate a new-document workflow

Suppose an AI assistant suggests calling `fit_transform` again on new documents. Predict why that can make the new vectors incompatible with the original matrix, then inspect the evidence below.

In [ ]:
new_documents = ["Researchers study data science."]

# Unsafe suggestion: fitting again creates a different vocabulary and column layout.
refit_vectorizer = CountVectorizer()
refit_matrix = refit_vectorizer.fit_transform(new_documents)

# Valid workflow: transform with the vectorizer fitted on the original corpus.
aligned_matrix = count_vectorizer.transform(new_documents)

workflow_check = pd.DataFrame({
    "workflow": ["refit on new text", "transform with fitted vectorizer"],
    "columns": [refit_matrix.shape[1], aligned_matrix.shape[1]],
    "matches original feature count": [
        refit_matrix.shape[1] == count_matrix.shape[1],
        aligned_matrix.shape[1] == count_matrix.shape[1],
    ],
})
assert aligned_matrix.shape[1] == count_matrix.shape[1]
workflow_check

**Explain:** Why must evaluation or future documents use `transform` rather than `fit_transform`? What happens to a word such as `researchers` that was not in the fitted vocabulary?

## 7. Independent practice: choose a representation

Run the starter analysis. Then choose **count**, **binary**, **unigram + bigram**, or **TF-IDF** for one plausible analysis of these clinical-workflow notes. Modify or add code to inspect the evidence you need.

In [ ]:
practice_documents = [
    "Doctors review patient notes.",
    "Nurses review clinical notes.",
    "Doctors and nurses analyze patient data.",
]

practice_models = {
    "count": CountVectorizer(),
    "binary": CountVectorizer(binary=True),
    "unigram + bigram": CountVectorizer(ngram_range=(1, 2)),
    "TF-IDF": TfidfVectorizer(),
}

practice_summary = []
for name, model in practice_models.items():
    matrix = model.fit_transform(practice_documents)
    practice_summary.append({
        "representation": name,
        "documents": matrix.shape[0],
        "features": matrix.shape[1],
        "nonzero values": matrix.nnz,
    })

pd.DataFrame(practice_summary).set_index("representation")

In [ ]:
# Student response area: these fields are intentionally blank.
chosen_representation = ""
justification = ""
evidence_from_output = ""
information_lost = ""

Use your code and observations to answer:

1. Which representation did you choose, and what analysis is it suitable for?
2. Cite one matrix value, feature, or shape as evidence.
3. What useful information does your chosen representation lose?
4. How would one preprocessing change from Week 02 alter the vocabulary or values?

## 8. Bridge to embeddings

The representations above are **sparse** and **lexical**: their columns correspond to observable terms or n-grams, which makes them interpretable. They usually do not recognize that different words can have related meanings, and they cannot represent unseen vocabulary without a defined fallback.

Dense embeddings trade much of that direct column-level interpretability for compact features that can capture patterns of usage and semantic relatedness. This does not make Bag-of-Words or TF-IDF obsolete; the right representation depends on the task, data, need for interpretation, and validation evidence. Week 04 will use numerical representations for comparison and retrieval.

**Final reflection:** Identify one situation where you would prefer TF-IDF and one where an embedding may be more appropriate. State what evidence you would use to validate the choice.